# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Reading from bronze table

In [0]:
df = spark.table("workspace.bronze.erp_px_cat")

# Data transformations

## Renaming columns

In [0]:
RENAME_MAP = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Normalize maintenance flag to boolean

In [0]:

df = (
    df
    .withColumn(
        "maintenance_flag",
        F.when(F.upper(col("maintenance_flag")) == "YES", F.lit(True))
         .when(F.upper(col("maintenance_flag")) == "NO", F.lit(False))
         .otherwise(None)
    )
)

## Sanity check of final DataFrame

In [0]:
df.limit(10).display()

# Write into silver table

In [0]:
(
    df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("workspace.silver.erp_product_category")
)